# 10 — TMDB Movie × Keyword Pairs (SCL Enriched)

Joins the movie–keyword edge list (`03`) with the SCL-enriched keyword table (`09`).
The result is a **flat, analysis-ready table** — one row per (movie, keyword) pair — with
sentiment/valence data attached.

**Inputs**
- `data/tmdb_movie_keyword_pairs.csv` — 928,781 (movie, keyword) edges
- `output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv` — 84,842 keywords with taxonomy + SCL valence

**Output**
- `data/tmdb_movie_keyword_pairs_scl_enriched.csv`
- `output/tmdb-movie-keyword-pairs-scl-enriched/`

**Columns added (from keyword table)**

| Column | Description |
|---|---|
| `name` | Keyword text |
| `keyword_type` | Classifier label |
| `is_narrative` | True = content keyword (not production/genre meta) |
| `movie_count` | How many movies use this keyword |
| `scl_valence` | Hierarchy-selected valence (OPP → NMA → VAD) |
| `scl_valence_source` | Which lexicon (`vad` / `opp` / `nma`) |
| `scl_polarity` | positive / negative / neutral / unscored |
| `scl_is_scored` | Has any valence score |
| `scl_is_phrase_level` | Score came from OPP or NMA (phrase-level) |

In [1]:
import shutil
from pathlib import Path

import pandas as pd

PAIRS_FILE   = Path("data/tmdb_movie_keyword_pairs.csv")
KW_FILE      = Path("output/tmdb-keyword-enriched/tmdb_keywords_enriched.csv")
OUT_FILE     = Path("data/tmdb_movie_keyword_pairs_scl_enriched.csv")
OUTPUT_DIR   = Path("output/tmdb-movie-keyword-pairs-scl-enriched")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
pairs = pd.read_csv(PAIRS_FILE)
kw    = pd.read_csv(KW_FILE)

print(f"Pairs    : {len(pairs):,}  columns: {pairs.columns.tolist()}")
print(f"Keywords : {len(kw):,}  columns: {kw.columns.tolist()}")

Pairs    : 928,781  columns: ['tmdb_movie_id', 'tmdb_keyword_id']
Keywords : 84,842  columns: ['tmdb_keyword_id', 'name', 'keyword_type', 'lexical_class', 'min_zipf_freq', 'is_narrative', 'movie_count', 'manually_reviewed', 'scl_valence', 'scl_valence_source', 'scl_in_vad', 'scl_in_opp', 'scl_in_nma', 'scl_composition_type', 'scl_composition_shift', 'scl_shift_zscore', 'scl_shift_percentile', 'scl_is_scored', 'scl_is_phrase_level', 'scl_polarity', 'scl_generated_coverage']


In [3]:
# Columns to bring in from keyword table
KW_COLS = [
    "tmdb_keyword_id",
    "name",
    "keyword_type",
    "is_narrative",
    "movie_count",
    "scl_valence",
    "scl_valence_source",
    "scl_polarity",
    "scl_is_scored",
    "scl_is_phrase_level",
]

enriched = pairs.merge(kw[KW_COLS], on="tmdb_keyword_id", how="left")

print(f"After join: {enriched.shape}")
print(f"Rows with valence: {enriched['scl_valence'].notna().sum():,} / {len(enriched):,} "
      f"({enriched['scl_valence'].notna().mean()*100:.1f}%)")

After join: (928781, 11)
Rows with valence: 786,996 / 928,781 (84.7%)


In [4]:
print("Polarity breakdown (all pairs):")
print(enriched["scl_polarity"].value_counts())
print()
print("Polarity breakdown (narrative keywords only):")
print(enriched.loc[enriched["is_narrative"], "scl_polarity"].value_counts())

Polarity breakdown (all pairs):
scl_polarity
positive    498223
negative    273351
unscored    141785
neutral      15422
Name: count, dtype: int64

Polarity breakdown (narrative keywords only):
scl_polarity
positive    370151
negative    258876
unscored    129927
neutral      14825
Name: count, dtype: int64


In [5]:
print(enriched.dtypes)
enriched.head(5)

tmdb_movie_id            int64
tmdb_keyword_id          int64
name                       str
keyword_type               str
is_narrative              bool
movie_count              int64
scl_valence            float64
scl_valence_source         str
scl_polarity               str
scl_is_scored             bool
scl_is_phrase_level       bool
dtype: object


,tmdb_movie_id,tmdb_keyword_id,name,keyword_type,is_narrative,movie_count,scl_valence,scl_valence_source,scl_polarity,scl_is_scored,scl_is_phrase_level
0,2,240,underdog,theme_or_plot,True,104,-0.5000,vad,negative,True,False
1,2,378,prison,location,True,1110,-0.8890,nma,negative,True,True
2,2,730,factory worker,theme_or_plot,True,106,0.2190,unigram_avg,positive,True,False
3,2,1563,prisoner,character_role,True,236,-0.5560,nma,negative,True,True
4,2,13072,falling in love,plot_event,True,212,0.1865,unigram_avg,positive,True,False


In [6]:
enriched.to_csv(OUT_FILE, index=False)
shutil.copy(OUT_FILE, OUTPUT_DIR / OUT_FILE.name)
print(f"Saved {len(enriched):,} rows → {OUT_FILE}")
print(f"Columns: {enriched.columns.tolist()}")

Saved 928,781 rows → data/tmdb_movie_keyword_pairs_scl_enriched.csv
Columns: ['tmdb_movie_id', 'tmdb_keyword_id', 'name', 'keyword_type', 'is_narrative', 'movie_count', 'scl_valence', 'scl_valence_source', 'scl_polarity', 'scl_is_scored', 'scl_is_phrase_level']


In [7]:
import os
from datetime import date

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub
    user  = kagglehub.whoami(verbose=False)["username"]
    notes = f"Pipeline run {date.today()}"
    slug  = "tmdb-movie-keyword-pairs-scl-enriched"
    print(f"Uploading {slug} ...")
    kagglehub.dataset_upload(
        handle=f"{user}/{slug}",
        local_dataset_dir=str(OUTPUT_DIR),
        version_notes=notes,
    )
    print(f"  → kaggle.com/datasets/{user}/{slug}")

Skipping upload: set KAGGLE_TOKEN env var to upload.
